<table align="left">
  <td>
    <a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" title="Abrir y ejecutar en Google Colaboratory"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Abrir en Kaggle" title="Abrir y ejecutar en Kaggle"/></a>
  </td>
</table>

<br><br>

# Series de tiempo, forecasting tabular y walk-forward validation

**Curso:** Aprendizaje Automático — SI7009 - 1 (5553)  
**Sesión 3:** Series de tiempo, forecasting tabular y walk-forward validation  
**Universidad:** EAFIT  
**Profesor:** Marco Teran  
**Dataset canónico:** Metro Interstate Traffic Volume  
**Dataset de respaldo:** Bike Sharing Dataset

---

## Propósito del notebook

Este notebook materializa la lógica metodológica de la Sesión 3:

1. formular target temporal, momento de predicción y horizonte;
2. transformar una serie temporal en una tabla supervisada;
3. construir lags, rolling windows, calendario y codificación cíclica;
4. evaluar baselines temporales antes de Machine Learning;
5. demostrar random split como anti-ejemplo;
6. implementar temporal holdout y walk-forward validation;
7. calcular MAE, RMSE y MASE;
8. evaluar horizontes \(h \in \{1,3,6,24\}\);
9. reservar test hasta el final;
10. cerrar con auditoría conceptual, metodológica y de leakage.

La pregunta central no es “qué modelo da el número más bajo”, sino:

> ¿Qué resultado temporal puedo creer si las filas tienen pasado, presente y futuro?

## Tabla de contenidos

1. Contrato metodológico de la sesión  
2. Conceptos previos reactivados  
3. Setup reproducible e instalación  
4. Imports, configuración y dependencias opcionales  
5. Funciones auxiliares  
6. Dummy manual: alineación lag-target  
7. Serie sintética: tendencia, estacionalidad y ruido  
8. Codificación cíclica  
9. Dataset real: Metro/Bike/fallback operativo  
10. Auditoría inicial de la serie  
11. Serie a tabla supervisada  
12. Split temporal train/validation/test  
13. Baselines temporales  
14. Ridge Regression e HistGradientBoostingRegressor  
15. Comparación holdout temporal  
16. Walk-forward validation  
17. Random split como anti-ejemplo  
18. Evaluación multi-horizon  
19. Test temporal reservado  
20. Extensiones opcionales: ARIMA/SARIMA/XGBoost/LightGBM  
21. Registro final de evidencia  
22. Matriz de validación conceptual  
23. Auditoría final  
24. Ejercicios y cierre

## 1. Contrato metodológico de la sesión

Reglas del notebook:

- El tiempo no se trata como una columna ordinaria.
- Cada fila supervisada representa una predicción hecha en un momento \(t\).
- Las features deben estar disponibles antes o en el momento de predicción.
- El target pertenece al futuro: \(y_{t+h}\).
- Los lags y rolling windows miran hacia atrás.
- El test temporal no se usa para escoger modelos, features, hiperparámetros ni decisiones.
- Random split se usa solo como anti-ejemplo.
- Los modelos compiten contra baselines temporales honestos.
- MASE responde si el modelo supera una predicción naïve.
- El desempeño se reporta por horizonte cuando corresponde.

In [ ]:
# =============================================================================
# 1.1 Methodological contract
# =============================================================================

methodological_contract = {
    "course": "Aprendizaje Automático — SI7009 - 1 (5553)",
    "session": "Sesión 3",
    "session_title": "Series de tiempo, forecasting tabular y walk-forward validation",
    "primary_task": "tabular_forecasting",
    "test_policy": "reserved_until_final_temporal_evaluation",
    "target_policy": "future_target_y_t_plus_h",
    "feature_policy": "features_available_at_or_before_prediction_time_t",
    "validation_policy": "temporal_holdout_and_walk_forward",
    "random_split_policy": "anti_example_only",
    "baseline_policy": "temporal_baselines_before_ml_models",
    "metrics": ["MAE", "RMSE", "MASE"],
    "horizons": [1, 3, 6, 24],
}

methodological_contract

## 2. Conceptos previos reactivados

| Concepto previo | Cómo se reactiva aquí | Por qué importa |
|---|---|---|
| baseline | naïve, seasonal naïve y moving average | Un modelo temporal debe superar referencias simples |
| train/validation/test | split cronológico | El futuro no puede contaminar la selección |
| leakage | temporal leakage | La disponibilidad de información depende de \(t\) |
| métricas | MAE, RMSE, MASE | El error debe tener unidades y contexto |
| pipelines | preprocessing dentro de train | Evita aprender estadísticas del futuro |
| comparación justa | mismo horizonte y protocolo | El ganador depende de la formulación |
| overfitting | estabilidad entre folds temporales | Un buen resultado en un periodo puede no generalizar |

## 3. Setup reproducible e instalación

Librerías requeridas:

- `numpy`
- `pandas`
- `matplotlib`
- `scikit-learn`

Librerías opcionales:

- `statsmodels`: referencia ARIMA/SARIMA.
- `xgboost`: extensión opcional.
- `lightgbm`: extensión opcional.

Instalación opcional:

```bash
pip install -U numpy pandas matplotlib scikit-learn statsmodels xgboost lightgbm
```

El flujo central no depende de `statsmodels`, `xgboost` ni `lightgbm`.

In [ ]:
# =============================================================================
# 3.1 Optional installation cell
# =============================================================================

# Uncomment only if needed.
# !pip install -U -q numpy pandas matplotlib scikit-learn statsmodels xgboost lightgbm

In [ ]:
# =============================================================================
# 4.1 Imports and global configuration
# =============================================================================

from __future__ import annotations

import math
import os
import sys
import time
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FAST_DEMO_MODE = True

MAIN_HORIZON = 1
HORIZONS = [1, 3, 6, 24]
SEASONAL_PERIOD = 24

TEST_FRACTION = 0.15
VALIDATION_FRACTION = 0.15

TARGET_COL_REAL = "traffic_volume"
DATETIME_COL = "datetime"

LAGS = [1, 2, 3, 24, 48, 168]
ROLLING_WINDOWS = [3, 6, 24]

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

results_registry: Dict[str, Any] = {}
audit_registry: Dict[str, Any] = {}
model_registry: Dict[str, Any] = {}

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print("Setup base importado correctamente.")

In [ ]:
# =============================================================================
# 4.2 Optional package availability
# =============================================================================

STATSMODELS_AVAILABLE = False
XGBOOST_AVAILABLE = False
LIGHTGBM_AVAILABLE = False

ARIMA = None
SARIMAX = None
XGBRegressor = None
LGBMRegressor = None

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATSMODELS_AVAILABLE = True
except Exception as exc:
    print(f"statsmodels no disponible. Se omiten ARIMA/SARIMA opcionales. Detalle: {exc}")

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception as exc:
    print(f"XGBoost no disponible. Se omite extensión opcional. Detalle: {exc}")

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    print(f"LightGBM no disponible. Se omite extensión opcional. Detalle: {exc}")

package_availability = pd.DataFrame(
    [
        {"package": "numpy", "required": True, "available": True},
        {"package": "pandas", "required": True, "available": True},
        {"package": "matplotlib", "required": True, "available": True},
        {"package": "scikit-learn", "required": True, "available": True},
        {"package": "statsmodels", "required": False, "available": STATSMODELS_AVAILABLE},
        {"package": "xgboost", "required": False, "available": XGBOOST_AVAILABLE},
        {"package": "lightgbm", "required": False, "available": LIGHTGBM_AVAILABLE},
    ]
)

results_registry["package_availability"] = package_availability

display(package_availability)

## 5. Funciones auxiliares

Estas funciones hacen que el notebook sea mantenible y auditable:

- validación de columnas;
- target futuro;
- lags;
- rolling windows alineadas con `shift(1)`;
- codificación cíclica;
- MAE, RMSE y MASE;
- split temporal;
- walk-forward validation;
- auditorías finales.

In [ ]:
# =============================================================================
# 5.1 General utilities
# =============================================================================

def print_section(title: str, width: int = 96) -> None:
    """Print a readable section header."""
    print("\n" + "=" * width)
    print(title.center(width))
    print("=" * width)


def require_columns(
    dataframe: pd.DataFrame,
    required_columns: Sequence[str],
    dataframe_name: str = "dataframe",
) -> None:
    """Raise a clear error if required columns are missing."""
    missing = [column for column in required_columns if column not in dataframe.columns]

    if missing:
        raise ValueError(
            f"{dataframe_name} is missing required columns: {missing}. "
            f"Available columns: {list(dataframe.columns)}"
        )


def audit_dataframe_schema(
    dataframe: pd.DataFrame,
    datetime_col: Optional[str] = None,
    target_col: Optional[str] = None,
) -> pd.DataFrame:
    """Create a compact schema audit table."""
    rows = []

    for column in dataframe.columns:
        rows.append(
            {
                "column": column,
                "dtype": str(dataframe[column].dtype),
                "missing_count": int(dataframe[column].isna().sum()),
                "missing_rate": float(dataframe[column].isna().mean()),
                "is_datetime": column == datetime_col,
                "is_target": column == target_col,
            }
        )

    return pd.DataFrame(rows)


def root_mean_squared_error_compatible(
    y_true: Sequence[float],
    y_pred: Sequence[float],
) -> float:
    """Compute RMSE with compatibility across scikit-learn versions."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mase(
    y_true: Sequence[float],
    y_pred: Sequence[float],
    y_train_scale: Sequence[float],
    seasonal_period: int = 1,
    epsilon: float = 1e-12,
) -> float:
    """Compute Mean Absolute Scaled Error from scratch."""
    y_true_array = np.asarray(y_true, dtype=float)
    y_pred_array = np.asarray(y_pred, dtype=float)
    y_train_array = np.asarray(y_train_scale, dtype=float)

    if len(y_train_array) <= seasonal_period:
        raise ValueError("y_train_scale must be longer than seasonal_period.")

    numerator = np.mean(np.abs(y_true_array - y_pred_array))
    denominator = np.mean(np.abs(y_train_array[seasonal_period:] - y_train_array[:-seasonal_period]))

    return float(numerator / max(denominator, epsilon))


def evaluate_regression_predictions(
    y_true: Sequence[float],
    y_pred: Sequence[float],
    y_train_scale: Sequence[float],
    model_name: str,
    dataset_name: str,
    horizon: int,
    validation_protocol: str,
    seasonal_period: int = 1,
) -> pd.DataFrame:
    """Evaluate regression predictions with MAE, RMSE and MASE."""
    mae_value = float(mean_absolute_error(y_true, y_pred))
    rmse_value = root_mean_squared_error_compatible(y_true, y_pred)
    mase_value = mase(y_true, y_pred, y_train_scale=y_train_scale, seasonal_period=seasonal_period)

    return pd.DataFrame(
        [
            {
                "dataset": dataset_name,
                "model": model_name,
                "horizon": horizon,
                "validation_protocol": validation_protocol,
                "mae": mae_value,
                "rmse": rmse_value,
                "mase": mase_value,
                "n_observations": len(y_true),
            }
        ]
    )


print("General utilities ready.")

In [ ]:
# =============================================================================
# 5.2 Time-series feature engineering utilities
# =============================================================================

def ensure_datetime_sorted(
    dataframe: pd.DataFrame,
    datetime_col: str,
) -> pd.DataFrame:
    """Return a copy sorted by datetime."""
    require_columns(dataframe, [datetime_col], "dataframe")

    output = dataframe.copy()
    output[datetime_col] = pd.to_datetime(output[datetime_col])
    output = output.sort_values(datetime_col).reset_index(drop=True)

    return output


def add_forecast_target(
    dataframe: pd.DataFrame,
    target_col: str,
    datetime_col: str,
    horizon: int,
) -> pd.DataFrame:
    """Create future target y_{t+h} for a forecasting horizon."""
    require_columns(dataframe, [target_col, datetime_col], "dataframe")

    if horizon <= 0:
        raise ValueError("horizon must be positive.")

    output = ensure_datetime_sorted(dataframe, datetime_col)
    output[f"target_h{horizon}"] = output[target_col].shift(-horizon)

    return output


def add_lag_features(
    dataframe: pd.DataFrame,
    target_col: str,
    lags: Sequence[int],
) -> pd.DataFrame:
    """Add lag features using only past values."""
    require_columns(dataframe, [target_col], "dataframe")

    output = dataframe.copy()

    for lag in lags:
        if lag <= 0:
            raise ValueError("All lags must be positive.")
        output[f"lag_{lag}"] = output[target_col].shift(lag)

    return output


def add_rolling_features(
    dataframe: pd.DataFrame,
    target_col: str,
    windows: Sequence[int],
) -> pd.DataFrame:
    """Add rolling features that look backward by shifting before rolling."""
    require_columns(dataframe, [target_col], "dataframe")

    output = dataframe.copy()
    shifted = output[target_col].shift(1)

    for window in windows:
        if window <= 1:
            raise ValueError("Rolling windows must be greater than 1.")

        output[f"rolling_mean_{window}"] = shifted.rolling(window=window, min_periods=window).mean()
        output[f"rolling_std_{window}"] = shifted.rolling(window=window, min_periods=window).std()

    return output


def add_calendar_features(
    dataframe: pd.DataFrame,
    datetime_col: str,
) -> pd.DataFrame:
    """Add calendar features known at prediction time."""
    require_columns(dataframe, [datetime_col], "dataframe")

    output = dataframe.copy()
    dt = pd.to_datetime(output[datetime_col])

    output["hour"] = dt.dt.hour
    output["dayofweek"] = dt.dt.dayofweek
    output["dayofmonth"] = dt.dt.day
    output["month"] = dt.dt.month
    output["is_weekend"] = output["dayofweek"].isin([5, 6]).astype(int)

    return output


def add_cyclical_features(
    dataframe: pd.DataFrame,
    column: str,
    period: int,
) -> pd.DataFrame:
    """Encode a periodic variable with sine and cosine."""
    require_columns(dataframe, [column], "dataframe")

    output = dataframe.copy()
    output[f"{column}_sin"] = np.sin(2 * np.pi * output[column] / period)
    output[f"{column}_cos"] = np.cos(2 * np.pi * output[column] / period)

    return output


def make_supervised_forecasting_frame(
    dataframe: pd.DataFrame,
    target_col: str,
    datetime_col: str,
    horizon: int,
    lags: Sequence[int],
    rolling_windows: Sequence[int],
    include_calendar: bool = True,
    include_cyclical: bool = True,
    drop_missing: bool = True,
) -> pd.DataFrame:
    """Convert a time series into a supervised tabular forecasting frame."""
    output = add_forecast_target(
        dataframe=dataframe,
        target_col=target_col,
        datetime_col=datetime_col,
        horizon=horizon,
    )

    output = add_lag_features(output, target_col=target_col, lags=lags)
    output = add_rolling_features(output, target_col=target_col, windows=rolling_windows)

    if include_calendar:
        output = add_calendar_features(output, datetime_col=datetime_col)

    if include_cyclical:
        if "hour" in output.columns:
            output = add_cyclical_features(output, "hour", 24)
        if "dayofweek" in output.columns:
            output = add_cyclical_features(output, "dayofweek", 7)
        if "month" in output.columns:
            output = add_cyclical_features(output, "month", 12)

    if drop_missing:
        output = output.dropna().reset_index(drop=True)

    return output


print("Time-series feature engineering utilities ready.")

In [ ]:
# =============================================================================
# 5.3 Split and plotting utilities
# =============================================================================

def temporal_train_validation_test_split(
    dataframe: pd.DataFrame,
    datetime_col: str,
    validation_fraction: float,
    test_fraction: float,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create chronological train, validation and test splits."""
    if validation_fraction + test_fraction >= 1:
        raise ValueError("validation_fraction + test_fraction must be less than 1.")

    data = ensure_datetime_sorted(dataframe, datetime_col)
    n_rows = len(data)

    test_size = int(np.ceil(n_rows * test_fraction))
    validation_size = int(np.ceil(n_rows * validation_fraction))
    train_size = n_rows - validation_size - test_size

    if train_size <= 0:
        raise ValueError("Not enough rows for train/validation/test split.")

    train = data.iloc[:train_size].copy()
    validation = data.iloc[train_size:train_size + validation_size].copy()
    test = data.iloc[train_size + validation_size:].copy()

    summary = pd.DataFrame(
        [
            {"split": "train", "rows": len(train), "start": train[datetime_col].min(), "end": train[datetime_col].max()},
            {"split": "validation", "rows": len(validation), "start": validation[datetime_col].min(), "end": validation[datetime_col].max()},
            {"split": "test", "rows": len(test), "start": test[datetime_col].min(), "end": test[datetime_col].max()},
        ]
    )

    summary["chronology_ok"] = [
        True,
        bool(train[datetime_col].max() < validation[datetime_col].min()),
        bool(validation[datetime_col].max() < test[datetime_col].min()),
    ]

    return train, validation, test, summary


def plot_time_series(
    dataframe: pd.DataFrame,
    datetime_col: str,
    target_col: str,
    title: str,
    max_points: Optional[int] = None,
) -> None:
    """Plot a time series."""
    data = ensure_datetime_sorted(dataframe, datetime_col)

    if max_points is not None and len(data) > max_points:
        data = data.tail(max_points)

    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.plot(data[datetime_col], data[target_col], linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel("Fecha")
    ax.set_ylabel(target_col)
    plt.show()


def plot_prediction_vs_observed(
    dates: Sequence[Any],
    y_true: Sequence[float],
    y_pred: Sequence[float],
    title: str,
    max_points: Optional[int] = 500,
) -> None:
    """Plot observed and predicted values over time."""
    plot_data = pd.DataFrame(
        {
            "datetime": pd.to_datetime(dates),
            "observed": np.asarray(y_true, dtype=float),
            "predicted": np.asarray(y_pred, dtype=float),
        }
    ).sort_values("datetime")

    if max_points is not None and len(plot_data) > max_points:
        plot_data = plot_data.tail(max_points)

    fig, ax = plt.subplots(figsize=(12, 5.5))
    ax.plot(plot_data["datetime"], plot_data["observed"], linewidth=2, label="Observado")
    ax.plot(plot_data["datetime"], plot_data["predicted"], linewidth=2, label="Predicho")
    ax.set_title(title)
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Valor")
    ax.legend()
    plt.show()


print("Split and plotting utilities ready.")

## 6. Dummy manual: alineación entre lag y target futuro

Este ejemplo pequeño muestra que:

- `lag_1` mira al pasado;
- `target_h1` mira al futuro;
- las primeras y últimas filas pueden perderse por falta de historia o target futuro.

In [ ]:
# =============================================================================
# 6.1 Manual toy time series
# =============================================================================

manual_series = pd.DataFrame(
    {
        "datetime": pd.date_range("2026-05-02 08:00:00", periods=8, freq="h"),
        "traffic_volume": [120, 135, 150, 142, 160, 175, 168, 180],
    }
)

manual_supervised = make_supervised_forecasting_frame(
    dataframe=manual_series,
    target_col="traffic_volume",
    datetime_col="datetime",
    horizon=1,
    lags=[1],
    rolling_windows=[3],
    include_calendar=True,
    include_cyclical=True,
    drop_missing=False,
)

manual_alignment_view = manual_supervised[
    ["datetime", "traffic_volume", "lag_1", "rolling_mean_3", "target_h1"]
].copy()

results_registry["manual_series"] = manual_series
results_registry["manual_alignment_view"] = manual_alignment_view

display(manual_alignment_view)

plot_time_series(
    dataframe=manual_series,
    datetime_col="datetime",
    target_col="traffic_volume",
    title="Dummy manual — serie original",
)

## 7. Serie sintética: tendencia, estacionalidad y ruido

La serie sintética sirve para visualizar estructura temporal controlada y splits cronológicos.

In [ ]:
# =============================================================================
# 7.1 Synthetic temporal dataset
# =============================================================================

def generate_synthetic_temporal_series(
    periods: int = 24 * 90,
    start: str = "2026-01-01",
    freq: str = "h",
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Generate a controlled temporal series with trend, seasonality and noise."""
    rng = np.random.default_rng(random_state)
    datetime_index = pd.date_range(start=start, periods=periods, freq=freq)

    t = np.arange(periods)
    daily_cycle = 25 * np.sin(2 * np.pi * t / 24)
    weekly_cycle = 12 * np.sin(2 * np.pi * t / (24 * 7))
    trend = 0.03 * t
    noise = rng.normal(0, 8, size=periods)

    traffic_volume = 220 + trend + daily_cycle + weekly_cycle + noise
    traffic_volume = np.maximum(traffic_volume, 10)

    return pd.DataFrame({"datetime": datetime_index, "traffic_volume": traffic_volume})


synthetic_temporal = generate_synthetic_temporal_series()

results_registry["synthetic_temporal"] = synthetic_temporal

display(synthetic_temporal.head())

plot_time_series(
    dataframe=synthetic_temporal,
    datetime_col="datetime",
    target_col="traffic_volume",
    title="Serie sintética — tendencia, ciclo diario, ciclo semanal y ruido",
    max_points=24 * 21,
)

## 8. Codificación cíclica

La hora del día es circular: 23:00 y 00:00 están cerca.

In [ ]:
# =============================================================================
# 8.1 Cyclical encoding demonstration
# =============================================================================

cyclical_encoding_table = pd.DataFrame({"hour": np.arange(24)})
cyclical_encoding_table = add_cyclical_features(cyclical_encoding_table, column="hour", period=24)

results_registry["cyclical_encoding_table"] = cyclical_encoding_table

display(cyclical_encoding_table.head(24))

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(cyclical_encoding_table["hour_cos"], cyclical_encoding_table["hour_sin"], s=80)

for _, row in cyclical_encoding_table.iterrows():
    ax.text(row["hour_cos"], row["hour_sin"], str(int(row["hour"])), fontsize=9)

ax.set_title("Codificación cíclica de hora del día")
ax.set_xlabel("hour_cos")
ax.set_ylabel("hour_sin")
ax.set_aspect("equal", adjustable="box")
plt.show()

## 9. Dataset real: Metro/Bike/fallback operativo

El notebook busca Metro Interstate Traffic Volume en rutas comunes.  
Si no aparece, busca Bike Sharing.  
Si tampoco aparece, genera un fallback sintético realista para preservar la ejecución.

El fallback no reemplaza el dataset real, pero mantiene la lógica metodológica de la clase.

In [ ]:
# =============================================================================
# 9.1 Dataset resolvers and fallback generator
# =============================================================================

def find_existing_file(candidate_paths: Sequence[Path]) -> Optional[Path]:
    """Return the first existing file in candidate paths."""
    for path in candidate_paths:
        if path.exists():
            return path
    return None


def find_metro_dataset() -> Optional[Path]:
    """Search common paths for Metro Interstate Traffic Volume."""
    candidate_paths = [
        Path("Metro_Interstate_Traffic_Volume.csv"),
        Path("metro_interstate_traffic_volume.csv"),
        Path("Metro_Interstate_Traffic_Volume.csv.gz"),
        Path("data/Metro_Interstate_Traffic_Volume.csv"),
        Path("data/metro_interstate_traffic_volume.csv"),
        Path("../input/metro-interstate-traffic-volume/Metro_Interstate_Traffic_Volume.csv"),
        Path("/kaggle/input/metro-interstate-traffic-volume/Metro_Interstate_Traffic_Volume.csv"),
        Path("/content/Metro_Interstate_Traffic_Volume.csv"),
        Path("/content/data/Metro_Interstate_Traffic_Volume.csv"),
    ]

    return find_existing_file(candidate_paths)


def find_bike_dataset() -> Optional[Path]:
    """Search common paths for Bike Sharing hourly data."""
    candidate_paths = [
        Path("hour.csv"),
        Path("bike_hour.csv"),
        Path("data/hour.csv"),
        Path("data/bike_hour.csv"),
        Path("../input/bike-sharing-dataset/hour.csv"),
        Path("/kaggle/input/bike-sharing-dataset/hour.csv"),
        Path("/content/hour.csv"),
        Path("/content/data/hour.csv"),
    ]

    return find_existing_file(candidate_paths)


def normalize_metro_dataframe(raw: pd.DataFrame) -> pd.DataFrame:
    """Normalize Metro dataset column names and datetime."""
    dataframe = raw.copy()
    dataframe.columns = [str(column).strip() for column in dataframe.columns]

    if "date_time" in dataframe.columns and "datetime" not in dataframe.columns:
        dataframe["datetime"] = pd.to_datetime(dataframe["date_time"])
    elif "datetime" in dataframe.columns:
        dataframe["datetime"] = pd.to_datetime(dataframe["datetime"])
    else:
        raise ValueError("Metro dataset requires date_time or datetime column.")

    if "traffic_volume" not in dataframe.columns:
        raise ValueError("Metro dataset requires traffic_volume column.")

    dataframe["traffic_volume"] = pd.to_numeric(dataframe["traffic_volume"], errors="coerce")

    return dataframe.dropna(subset=["datetime", "traffic_volume"]).sort_values("datetime").reset_index(drop=True)


def normalize_bike_dataframe(raw: pd.DataFrame) -> pd.DataFrame:
    """Normalize Bike Sharing hourly dataset to the notebook schema."""
    dataframe = raw.copy()
    dataframe.columns = [str(column).strip() for column in dataframe.columns]

    if "dteday" in dataframe.columns and "hr" in dataframe.columns:
        dataframe["datetime"] = pd.to_datetime(dataframe["dteday"]) + pd.to_timedelta(dataframe["hr"], unit="h")
    elif "datetime" in dataframe.columns:
        dataframe["datetime"] = pd.to_datetime(dataframe["datetime"])
    else:
        raise ValueError("Bike dataset requires dteday + hr or datetime columns.")

    if "cnt" in dataframe.columns:
        dataframe["traffic_volume"] = pd.to_numeric(dataframe["cnt"], errors="coerce")
    elif "count" in dataframe.columns:
        dataframe["traffic_volume"] = pd.to_numeric(dataframe["count"], errors="coerce")
    else:
        raise ValueError("Bike dataset requires cnt or count column.")

    return dataframe.dropna(subset=["datetime", "traffic_volume"]).sort_values("datetime").reset_index(drop=True)


def generate_realistic_traffic_fallback(
    periods: int = 24 * 180,
    start: str = "2018-01-01",
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Generate a realistic fallback traffic-like hourly dataset."""
    rng = np.random.default_rng(random_state)
    datetime_index = pd.date_range(start=start, periods=periods, freq="h")

    hour = datetime_index.hour.to_numpy()
    dayofweek = datetime_index.dayofweek.to_numpy()
    month = datetime_index.month.to_numpy()

    morning_peak = 900 * np.exp(-0.5 * ((hour - 8) / 2.0) ** 2)
    evening_peak = 1_050 * np.exp(-0.5 * ((hour - 17) / 2.5) ** 2)
    night_low = -350 * np.exp(-0.5 * ((hour - 3) / 2.0) ** 2)

    weekday_effect = np.where(dayofweek < 5, 450, -250)
    monthly_effect = 120 * np.sin(2 * np.pi * month / 12)
    trend = np.linspace(0, 180, periods)
    noise = rng.normal(0, 120, periods)

    traffic_volume = 1_900 + morning_peak + evening_peak + night_low + weekday_effect + monthly_effect + trend + noise
    traffic_volume = np.maximum(traffic_volume, 50)

    return pd.DataFrame(
        {
            "datetime": datetime_index,
            "traffic_volume": traffic_volume,
            "temp": 285 + 8 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 2, periods),
            "rain_1h": np.maximum(rng.gamma(shape=0.4, scale=0.8, size=periods) - 0.3, 0),
            "snow_1h": 0.0,
            "clouds_all": np.clip(50 + 25 * np.sin(2 * np.pi * month / 12) + rng.normal(0, 15, periods), 0, 100),
            "weather_main": np.where(rng.random(periods) < 0.15, "Rain", "Clear"),
            "holiday": np.where((dayofweek == 6) & (rng.random(periods) < 0.04), "Holiday", "None"),
        }
    )


def load_temporal_dataset() -> Tuple[pd.DataFrame, str, str]:
    """Load Metro, Bike fallback, or synthetic operational fallback."""
    metro_path = find_metro_dataset()

    if metro_path is not None:
        raw = pd.read_csv(metro_path)
        return normalize_metro_dataframe(raw), "Metro Interstate Traffic Volume", str(metro_path)

    bike_path = find_bike_dataset()

    if bike_path is not None:
        raw = pd.read_csv(bike_path)
        return normalize_bike_dataframe(raw), "Bike Sharing Dataset fallback", str(bike_path)

    fallback = generate_realistic_traffic_fallback()
    return fallback, "Synthetic traffic-like operational fallback", "generated_in_notebook"


temporal_raw, DATASET_NAME, DATASET_SOURCE = load_temporal_dataset()

results_registry["temporal_raw"] = temporal_raw
results_registry["dataset_name"] = DATASET_NAME
results_registry["dataset_source"] = DATASET_SOURCE

print(f"Dataset loaded: {DATASET_NAME}")
print(f"Dataset source: {DATASET_SOURCE}")
print(f"Shape: {temporal_raw.shape}")

display(temporal_raw.head())

## 10. Auditoría inicial de la serie

Antes de modelar revisamos:

- columnas;
- faltantes;
- rango temporal;
- frecuencia aproximada;
- duplicados;
- perfiles por hora y día.

In [ ]:
# =============================================================================
# 10.1 Dataset schema and temporal audit
# =============================================================================

temporal_raw = ensure_datetime_sorted(temporal_raw, DATETIME_COL)

schema_audit = audit_dataframe_schema(
    dataframe=temporal_raw,
    datetime_col=DATETIME_COL,
    target_col=TARGET_COL_REAL,
)

temporal_gap_summary = (
    temporal_raw[DATETIME_COL]
    .diff()
    .value_counts()
    .head(10)
    .reset_index()
)
temporal_gap_summary.columns = ["time_delta", "count"]

duplicate_datetime_count = int(temporal_raw[DATETIME_COL].duplicated().sum())

dataset_audit_summary = pd.DataFrame(
    [
        {
            "dataset": DATASET_NAME,
            "source": DATASET_SOURCE,
            "rows": len(temporal_raw),
            "columns": temporal_raw.shape[1],
            "start": temporal_raw[DATETIME_COL].min(),
            "end": temporal_raw[DATETIME_COL].max(),
            "duplicate_datetimes": duplicate_datetime_count,
            "target_missing": int(temporal_raw[TARGET_COL_REAL].isna().sum()),
            "target_min": float(temporal_raw[TARGET_COL_REAL].min()),
            "target_mean": float(temporal_raw[TARGET_COL_REAL].mean()),
            "target_max": float(temporal_raw[TARGET_COL_REAL].max()),
        }
    ]
)

results_registry["schema_audit"] = schema_audit
results_registry["temporal_gap_summary"] = temporal_gap_summary
results_registry["dataset_audit_summary"] = dataset_audit_summary

display(dataset_audit_summary)
display(schema_audit)
display(temporal_gap_summary)

plot_time_series(
    dataframe=temporal_raw,
    datetime_col=DATETIME_COL,
    target_col=TARGET_COL_REAL,
    title=f"{DATASET_NAME} — serie temporal del target",
    max_points=24 * 30,
)

In [ ]:
# =============================================================================
# 10.2 Calendar profiles
# =============================================================================

calendar_profile_frame = add_calendar_features(temporal_raw[[DATETIME_COL, TARGET_COL_REAL]].copy(), DATETIME_COL)

hourly_profile = calendar_profile_frame.groupby("hour", as_index=False)[TARGET_COL_REAL].mean()
weekday_profile = calendar_profile_frame.groupby("dayofweek", as_index=False)[TARGET_COL_REAL].mean()

results_registry["hourly_profile"] = hourly_profile
results_registry["weekday_profile"] = weekday_profile

display(hourly_profile)
display(weekday_profile)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(hourly_profile["hour"], hourly_profile[TARGET_COL_REAL], marker="o", linewidth=2)
ax.set_title("Perfil promedio por hora")
ax.set_xlabel("Hora")
ax.set_ylabel("Promedio del target")
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(weekday_profile["dayofweek"], weekday_profile[TARGET_COL_REAL], alpha=0.85)
ax.set_title("Perfil promedio por día de la semana")
ax.set_xlabel("Día de la semana, 0 = lunes")
ax.set_ylabel("Promedio del target")
plt.show()

## 11. Serie a tabla supervisada

Para \(h=1\), cada fila contiene información disponible hasta \(t\) y target futuro \(y_{t+1}\).

In [ ]:
# =============================================================================
# 11.1 Build supervised forecasting frame
# =============================================================================

supervised_frame_real = make_supervised_forecasting_frame(
    dataframe=temporal_raw,
    target_col=TARGET_COL_REAL,
    datetime_col=DATETIME_COL,
    horizon=MAIN_HORIZON,
    lags=LAGS,
    rolling_windows=ROLLING_WINDOWS,
    include_calendar=True,
    include_cyclical=True,
    drop_missing=True,
)

target_h_col = f"target_h{MAIN_HORIZON}"

excluded_from_features = {
    DATETIME_COL,
    TARGET_COL_REAL,
    target_h_col,
    "date_time",
    "dteday",
    "dataset_source",
}

feature_columns_real = [
    column for column in supervised_frame_real.columns
    if column not in excluded_from_features
]

numeric_feature_columns_real = [
    column for column in feature_columns_real
    if pd.api.types.is_numeric_dtype(supervised_frame_real[column])
]

categorical_feature_columns_real = [
    column for column in feature_columns_real
    if not pd.api.types.is_numeric_dtype(supervised_frame_real[column])
]

supervised_frame_report = pd.DataFrame(
    [
        {
            "dataset": DATASET_NAME,
            "horizon": MAIN_HORIZON,
            "rows_after_framing": len(supervised_frame_real),
            "total_features": len(feature_columns_real),
            "numeric_features": len(numeric_feature_columns_real),
            "categorical_features": len(categorical_feature_columns_real),
            "target_column": target_h_col,
            "current_target_excluded": TARGET_COL_REAL not in feature_columns_real,
            "future_target_excluded": target_h_col not in feature_columns_real,
        }
    ]
)

results_registry["supervised_frame_real"] = supervised_frame_real
results_registry["feature_columns_real"] = feature_columns_real
results_registry["numeric_feature_columns_real"] = numeric_feature_columns_real
results_registry["categorical_feature_columns_real"] = categorical_feature_columns_real
results_registry["supervised_frame_report"] = supervised_frame_report

display(supervised_frame_report)
display(supervised_frame_real.head())

In [ ]:
# =============================================================================
# 11.2 Feature alignment and temporal leakage audit
# =============================================================================

alignment_audit_real = pd.DataFrame(
    [
        {
            "feature": "lag_1",
            "meaning": "valor inmediatamente anterior",
            "present": "lag_1" in supervised_frame_real.columns,
            "must_not_use_future": True,
            "status": "valid_if_shift_positive",
        },
        {
            "feature": "lag_24",
            "meaning": "misma hora del día anterior",
            "present": "lag_24" in supervised_frame_real.columns,
            "must_not_use_future": True,
            "status": "valid_if_shift_positive",
        },
        {
            "feature": "rolling_mean_24",
            "meaning": "promedio de 24 observaciones previas",
            "present": "rolling_mean_24" in supervised_frame_real.columns,
            "must_not_use_future": True,
            "status": "valid_because_shift_1_before_rolling",
        },
        {
            "feature": "hour_sin/hour_cos",
            "meaning": "representación circular de la hora",
            "present": {"hour_sin", "hour_cos"}.issubset(set(supervised_frame_real.columns)),
            "must_not_use_future": True,
            "status": "valid_calendar_known_at_prediction_time",
        },
        {
            "feature": target_h_col,
            "meaning": "target futuro",
            "present": target_h_col in supervised_frame_real.columns,
            "must_not_use_future": False,
            "status": "must_not_be_in_feature_columns",
        },
    ]
)

alignment_audit_real["passed"] = [
    bool(alignment_audit_real.loc[0, "present"]),
    bool(alignment_audit_real.loc[1, "present"]),
    bool(alignment_audit_real.loc[2, "present"]),
    bool(alignment_audit_real.loc[3, "present"]),
    bool(target_h_col not in feature_columns_real),
]

results_registry["alignment_audit_real"] = alignment_audit_real

display(alignment_audit_real)

if not alignment_audit_real["passed"].all():
    raise RuntimeError("Feature alignment audit failed.")

## 12. Split temporal train/validation/test

El split se hace por orden cronológico.

In [ ]:
# =============================================================================
# 12.1 Temporal train/validation/test split
# =============================================================================

train_frame_real, valid_frame_real, test_frame_real, temporal_split_summary_real = temporal_train_validation_test_split(
    dataframe=supervised_frame_real,
    datetime_col=DATETIME_COL,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
)

results_registry["train_frame_real"] = train_frame_real
results_registry["valid_frame_real"] = valid_frame_real
results_registry["test_frame_real"] = test_frame_real
results_registry["temporal_split_summary_real"] = temporal_split_summary_real

display(temporal_split_summary_real)

if not temporal_split_summary_real["chronology_ok"].all():
    raise RuntimeError("Temporal split chronology audit failed.")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train_frame_real[DATETIME_COL], train_frame_real[TARGET_COL_REAL], label="Train")
ax.plot(valid_frame_real[DATETIME_COL], valid_frame_real[TARGET_COL_REAL], label="Validation")
ax.plot(test_frame_real[DATETIME_COL], test_frame_real[TARGET_COL_REAL], label="Test")
ax.set_title("Split temporal train / validation / test")
ax.set_xlabel("Fecha")
ax.set_ylabel(TARGET_COL_REAL)
ax.legend()
plt.show()

In [ ]:
# =============================================================================
# 12.2 Feature matrices
# =============================================================================

X_train_real = train_frame_real[feature_columns_real].copy()
y_train_real = train_frame_real[target_h_col].copy()

X_valid_real = valid_frame_real[feature_columns_real].copy()
y_valid_real = valid_frame_real[target_h_col].copy()

X_test_real = test_frame_real[feature_columns_real].copy()
y_test_real = test_frame_real[target_h_col].copy()

feature_matrix_summary = pd.DataFrame(
    [
        {"split": "train", "rows": X_train_real.shape[0], "features": X_train_real.shape[1], "target_rows": len(y_train_real)},
        {"split": "validation", "rows": X_valid_real.shape[0], "features": X_valid_real.shape[1], "target_rows": len(y_valid_real)},
        {"split": "test", "rows": X_test_real.shape[0], "features": X_test_real.shape[1], "target_rows": len(y_test_real)},
    ]
)

results_registry["feature_matrix_summary"] = feature_matrix_summary

display(feature_matrix_summary)

## 13. Baselines temporales

Evaluamos:

- naïve baseline;
- seasonal naïve baseline;
- moving average baseline.

In [ ]:
# =============================================================================
# 13.1 Temporal baseline predictions
# =============================================================================

def add_temporal_baseline_predictions(
    dataframe: pd.DataFrame,
    target_col: str,
    horizon: int,
    seasonal_period: int,
    moving_average_window: int,
    datetime_col: str,
) -> pd.DataFrame:
    """Add naïve, seasonal naïve and moving-average baseline predictions."""
    data = ensure_datetime_sorted(dataframe, datetime_col).copy()

    data[f"naive_pred_h{horizon}"] = data[target_col]
    data[f"seasonal_naive_pred_h{horizon}"] = data[target_col].shift(seasonal_period - horizon)
    data[f"moving_average_pred_h{horizon}"] = (
        data[target_col]
        .shift(1)
        .rolling(window=moving_average_window, min_periods=moving_average_window)
        .mean()
    )

    return data


baseline_frame_real = add_temporal_baseline_predictions(
    dataframe=supervised_frame_real,
    target_col=TARGET_COL_REAL,
    horizon=MAIN_HORIZON,
    seasonal_period=SEASONAL_PERIOD,
    moving_average_window=24,
    datetime_col=DATETIME_COL,
).dropna(
    subset=[
        target_h_col,
        f"naive_pred_h{MAIN_HORIZON}",
        f"seasonal_naive_pred_h{MAIN_HORIZON}",
        f"moving_average_pred_h{MAIN_HORIZON}",
    ]
)

baseline_train_frame, baseline_valid_frame, baseline_test_frame, baseline_split_summary = temporal_train_validation_test_split(
    dataframe=baseline_frame_real,
    datetime_col=DATETIME_COL,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
)

baseline_name_to_col = {
    "naive_baseline": f"naive_pred_h{MAIN_HORIZON}",
    "seasonal_naive_baseline": f"seasonal_naive_pred_h{MAIN_HORIZON}",
    "moving_average_24_baseline": f"moving_average_pred_h{MAIN_HORIZON}",
}


def evaluate_temporal_baselines(
    train_frame: pd.DataFrame,
    valid_frame: pd.DataFrame,
    target_col: str,
    horizon: int,
    dataset_name: str,
    seasonal_period: int,
    validation_protocol: str,
) -> pd.DataFrame:
    """Evaluate temporal baselines."""
    reports = []

    for baseline_name, pred_col in baseline_name_to_col.items():
        report = evaluate_regression_predictions(
            y_true=valid_frame[f"target_h{horizon}"],
            y_pred=valid_frame[pred_col],
            y_train_scale=train_frame[target_col],
            model_name=baseline_name,
            dataset_name=dataset_name,
            horizon=horizon,
            validation_protocol=validation_protocol,
            seasonal_period=seasonal_period,
        )
        reports.append(report)

    return pd.concat(reports, ignore_index=True)


baseline_validation_report = evaluate_temporal_baselines(
    train_frame=baseline_train_frame,
    valid_frame=baseline_valid_frame,
    target_col=TARGET_COL_REAL,
    horizon=MAIN_HORIZON,
    dataset_name=DATASET_NAME,
    seasonal_period=SEASONAL_PERIOD,
    validation_protocol="temporal_holdout_validation",
)

results_registry["baseline_frame_real"] = baseline_frame_real
results_registry["baseline_validation_report"] = baseline_validation_report

display(baseline_validation_report.sort_values("mae"))

fig, ax = plt.subplots(figsize=(9, 5.2))
plot_df = baseline_validation_report.sort_values("mae")
ax.barh(plot_df["model"], plot_df["mae"], alpha=0.85)
ax.set_title("Baselines temporales — MAE en validation")
ax.set_xlabel("MAE")
ax.set_ylabel("Baseline")
plt.show()

## 14. Ridge Regression e HistGradientBoostingRegressor

Ridge Regression funciona como baseline supervisado interpretable.

HistGradientBoostingRegressor funciona como modelo tabular fuerte dentro de scikit-learn.

In [ ]:
# =============================================================================
# 14.1 Model builders
# =============================================================================

def build_preprocessor(
    numeric_features: Sequence[str],
    categorical_features: Sequence[str],
) -> ColumnTransformer:
    """Build preprocessing transformer for numeric and categorical features."""
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", encoder),
        ]
    )

    transformers = []

    if numeric_features:
        transformers.append(("numeric", numeric_pipeline, list(numeric_features)))

    if categorical_features:
        transformers.append(("categorical", categorical_pipeline, list(categorical_features)))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_ridge_pipeline(
    numeric_features: Sequence[str],
    categorical_features: Sequence[str],
    alpha: float = 1.0,
) -> Pipeline:
    """Build Ridge Regression pipeline."""
    preprocessor = build_preprocessor(numeric_features, categorical_features)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", Ridge(alpha=alpha)),
        ]
    )


def build_hist_gradient_boosting_model(
    max_iter: int = 300,
    learning_rate: float = 0.05,
    max_leaf_nodes: int = 31,
) -> HistGradientBoostingRegressor:
    """Build HistGradientBoostingRegressor."""
    return HistGradientBoostingRegressor(
        max_iter=max_iter,
        learning_rate=learning_rate,
        max_leaf_nodes=max_leaf_nodes,
        l2_regularization=0.0,
        random_state=RANDOM_STATE,
    )


ridge_pipeline = build_ridge_pipeline(
    numeric_features=numeric_feature_columns_real,
    categorical_features=categorical_feature_columns_real,
    alpha=1.0,
)

hgb_model = build_hist_gradient_boosting_model(
    max_iter=300 if FAST_DEMO_MODE else 600,
    learning_rate=0.05,
    max_leaf_nodes=31,
)

model_registry["ridge_regression"] = ridge_pipeline
model_registry["hist_gradient_boosting"] = hgb_model

model_table = pd.DataFrame(
    [
        {"model": "ridge_regression", "role": "baseline supervisado interpretable", "requires_preprocessing": True, "captures_nonlinearity": False},
        {"model": "hist_gradient_boosting", "role": "modelo tabular fuerte", "requires_preprocessing": False, "captures_nonlinearity": True},
    ]
)

results_registry["model_table"] = model_table

display(model_table)

In [ ]:
# =============================================================================
# 14.2 Fit, predict and evaluate supervised models
# =============================================================================

def fit_predict_evaluate_regressor(
    estimator: Any,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    y_train_scale: pd.Series,
    model_name: str,
    dataset_name: str,
    horizon: int,
    validation_protocol: str,
    seasonal_period: int,
) -> Tuple[pd.DataFrame, np.ndarray, Any]:
    """Fit a regressor, predict and evaluate."""
    fitted = clone(estimator)
    fitted.fit(X_train, y_train)

    predictions = fitted.predict(X_valid)

    report = evaluate_regression_predictions(
        y_true=y_valid,
        y_pred=predictions,
        y_train_scale=y_train_scale,
        model_name=model_name,
        dataset_name=dataset_name,
        horizon=horizon,
        validation_protocol=validation_protocol,
        seasonal_period=seasonal_period,
    )

    return report, predictions, fitted


supervised_reports = []
supervised_predictions: Dict[str, np.ndarray] = {}
fitted_validation_models: Dict[str, Any] = {}

for model_name, estimator in {"ridge_regression": ridge_pipeline, "hist_gradient_boosting": hgb_model}.items():
    report, predictions, fitted_model = fit_predict_evaluate_regressor(
        estimator=estimator,
        X_train=X_train_real,
        y_train=y_train_real,
        X_valid=X_valid_real,
        y_valid=y_valid_real,
        y_train_scale=train_frame_real[TARGET_COL_REAL],
        model_name=model_name,
        dataset_name=DATASET_NAME,
        horizon=MAIN_HORIZON,
        validation_protocol="temporal_holdout_validation",
        seasonal_period=SEASONAL_PERIOD,
    )

    supervised_reports.append(report)
    supervised_predictions[model_name] = predictions
    fitted_validation_models[model_name] = fitted_model

supervised_validation_report = pd.concat(supervised_reports, ignore_index=True)

results_registry["supervised_validation_report"] = supervised_validation_report
results_registry["supervised_predictions"] = supervised_predictions

display(supervised_validation_report.sort_values("mae"))

for model_name, predictions in supervised_predictions.items():
    plot_prediction_vs_observed(
        dates=valid_frame_real[DATETIME_COL],
        y_true=y_valid_real,
        y_pred=predictions,
        title=f"Validation temporal — {model_name}",
        max_points=24 * 14,
    )

## 15. Comparación holdout temporal

Combinamos baselines y modelos supervisados.

El test sigue intacto.

In [ ]:
# =============================================================================
# 15.1 Validation comparison and candidate selection
# =============================================================================

validation_comparison_report = pd.concat(
    [baseline_validation_report, supervised_validation_report],
    ignore_index=True,
).sort_values(["mae", "rmse"], ascending=[True, True]).reset_index(drop=True)

validation_comparison_report["rank_by_mae"] = validation_comparison_report["mae"].rank(method="min")
validation_comparison_report["test_set_used"] = False

results_registry["validation_comparison_report"] = validation_comparison_report

display(validation_comparison_report)

fig, ax = plt.subplots(figsize=(10, 5.4))
plot_df = validation_comparison_report.sort_values("mae", ascending=True)
ax.barh(plot_df["model"], plot_df["mae"], alpha=0.85)
ax.set_title("Comparación temporal holdout — MAE")
ax.set_xlabel("MAE")
ax.set_ylabel("Modelo")
plt.show()

candidate_row = validation_comparison_report.iloc[0]

CANDIDATE_MODEL_NAME = str(candidate_row["model"])
CANDIDATE_VALIDATION_MAE = float(candidate_row["mae"])
CANDIDATE_VALIDATION_RMSE = float(candidate_row["rmse"])
CANDIDATE_VALIDATION_MASE = float(candidate_row["mase"])

CANDIDATE_MODEL_TYPE = "temporal_baseline" if CANDIDATE_MODEL_NAME in baseline_name_to_col else "supervised_model"

candidate_selection_record = pd.DataFrame(
    [
        {
            "candidate_model": CANDIDATE_MODEL_NAME,
            "candidate_type": CANDIDATE_MODEL_TYPE,
            "selection_protocol": "temporal_holdout_validation",
            "selection_metric": "mae_then_rmse",
            "validation_mae": CANDIDATE_VALIDATION_MAE,
            "validation_rmse": CANDIDATE_VALIDATION_RMSE,
            "validation_mase": CANDIDATE_VALIDATION_MASE,
            "test_set_used": False,
        }
    ]
)

results_registry["candidate_selection_record"] = candidate_selection_record

display(candidate_selection_record)

## 16. Walk-forward validation

Walk-forward validation evalúa el modelo avanzando en el tiempo:

1. entrenar con pasado;
2. validar en futuro;
3. avanzar el origen;
4. resumir estabilidad.

Incluye un parámetro `embargo_size` para representar la idea de embargo cuando hay riesgo de contaminación por vecindad temporal.

In [ ]:
# =============================================================================
# 16.1 Walk-forward split generator and evaluator
# =============================================================================

def make_walk_forward_splits(
    dataframe: pd.DataFrame,
    datetime_col: str,
    initial_train_size: int,
    validation_size: int,
    step_size: Optional[int] = None,
    max_splits: Optional[int] = None,
    embargo_size: int = 0,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Create expanding-window walk-forward splits."""
    data = ensure_datetime_sorted(dataframe, datetime_col).reset_index(drop=True)

    if step_size is None:
        step_size = validation_size

    splits: List[Tuple[np.ndarray, np.ndarray]] = []
    train_end = initial_train_size

    while True:
        validation_start = train_end + embargo_size
        validation_end = validation_start + validation_size

        if validation_end > len(data):
            break

        train_idx = np.arange(0, train_end)
        valid_idx = np.arange(validation_start, validation_end)

        splits.append((train_idx, valid_idx))

        if max_splits is not None and len(splits) >= max_splits:
            break

        train_end += step_size

    return splits


def summarize_walk_forward_splits(
    dataframe: pd.DataFrame,
    splits: Sequence[Tuple[np.ndarray, np.ndarray]],
    datetime_col: str,
) -> pd.DataFrame:
    """Summarize walk-forward splits."""
    data = ensure_datetime_sorted(dataframe, datetime_col).reset_index(drop=True)
    rows = []

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        train_dates = data.loc[train_idx, datetime_col]
        valid_dates = data.loc[valid_idx, datetime_col]

        rows.append(
            {
                "fold": fold,
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
                "train_start": train_dates.min(),
                "train_end": train_dates.max(),
                "valid_start": valid_dates.min(),
                "valid_end": valid_dates.max(),
                "chronology_ok": bool(train_dates.max() < valid_dates.min()),
            }
        )

    return pd.DataFrame(rows)


def choose_walk_forward_sizes(n_rows: int, fast_demo_mode: bool = True) -> Tuple[int, int, int, int]:
    """Choose classroom-friendly walk-forward sizes."""
    if fast_demo_mode:
        initial_train_size = max(200, int(0.55 * n_rows))
        validation_size = max(24 * 7, int(0.08 * n_rows))
        max_splits = 4
    else:
        initial_train_size = max(300, int(0.45 * n_rows))
        validation_size = max(24 * 14, int(0.06 * n_rows))
        max_splits = 8

    step_size = validation_size

    return initial_train_size, validation_size, step_size, max_splits


wf_initial_train_size, wf_validation_size, wf_step_size, wf_max_splits = choose_walk_forward_sizes(
    n_rows=len(supervised_frame_real),
    fast_demo_mode=FAST_DEMO_MODE,
)

walk_forward_splits = make_walk_forward_splits(
    dataframe=supervised_frame_real,
    datetime_col=DATETIME_COL,
    initial_train_size=wf_initial_train_size,
    validation_size=wf_validation_size,
    step_size=wf_step_size,
    max_splits=wf_max_splits,
    embargo_size=0,
)

walk_forward_split_summary = summarize_walk_forward_splits(
    dataframe=supervised_frame_real,
    splits=walk_forward_splits,
    datetime_col=DATETIME_COL,
)

results_registry["walk_forward_splits"] = walk_forward_splits
results_registry["walk_forward_split_summary"] = walk_forward_split_summary

display(walk_forward_split_summary)

if walk_forward_split_summary.empty:
    raise RuntimeError("No walk-forward splits were created.")

if not walk_forward_split_summary["chronology_ok"].all():
    raise RuntimeError("Walk-forward chronology audit failed.")

In [ ]:
# =============================================================================
# 16.2 Walk-forward evaluation
# =============================================================================

def evaluate_model_walk_forward(
    dataframe: pd.DataFrame,
    splits: Sequence[Tuple[np.ndarray, np.ndarray]],
    datetime_col: str,
    feature_cols: Sequence[str],
    numeric_features: Sequence[str],
    categorical_features: Sequence[str],
    target_col_original: str,
    target_h_col: str,
    dataset_name: str,
    horizon: int,
    seasonal_period: int,
    fast_demo_mode: bool,
) -> pd.DataFrame:
    """Evaluate baselines and supervised models using walk-forward validation."""
    data = ensure_datetime_sorted(dataframe, datetime_col).reset_index(drop=True)

    baseline_data = add_temporal_baseline_predictions(
        dataframe=data,
        target_col=target_col_original,
        horizon=horizon,
        seasonal_period=seasonal_period,
        moving_average_window=24,
        datetime_col=datetime_col,
    )

    baseline_specs = [
        ("naive_baseline", f"naive_pred_h{horizon}"),
        ("seasonal_naive_baseline", f"seasonal_naive_pred_h{horizon}"),
        ("moving_average_24_baseline", f"moving_average_pred_h{horizon}"),
    ]

    reports = []

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        train_fold = baseline_data.iloc[train_idx].copy()
        valid_fold = baseline_data.iloc[valid_idx].copy()

        for baseline_name, pred_col in baseline_specs:
            fold_valid = valid_fold.dropna(subset=[target_h_col, pred_col]).copy()

            if fold_valid.empty:
                continue

            report = evaluate_regression_predictions(
                y_true=fold_valid[target_h_col],
                y_pred=fold_valid[pred_col],
                y_train_scale=train_fold[target_col_original],
                model_name=baseline_name,
                dataset_name=dataset_name,
                horizon=horizon,
                validation_protocol="walk_forward_validation",
                seasonal_period=seasonal_period,
            )

            report["fold"] = fold
            reports.append(report)

        X_train_fold = train_fold[list(feature_cols)].copy()
        y_train_fold = train_fold[target_h_col].copy()
        X_valid_fold = valid_fold[list(feature_cols)].copy()
        y_valid_fold = valid_fold[target_h_col].copy()

        fold_models = {
            "ridge_regression": build_ridge_pipeline(numeric_features, categorical_features, alpha=1.0),
            "hist_gradient_boosting": build_hist_gradient_boosting_model(
                max_iter=250 if fast_demo_mode else 500,
                learning_rate=0.05,
                max_leaf_nodes=31,
            ),
        }

        for model_name, estimator in fold_models.items():
            report, _, _ = fit_predict_evaluate_regressor(
                estimator=estimator,
                X_train=X_train_fold,
                y_train=y_train_fold,
                X_valid=X_valid_fold,
                y_valid=y_valid_fold,
                y_train_scale=train_fold[target_col_original],
                model_name=model_name,
                dataset_name=dataset_name,
                horizon=horizon,
                validation_protocol="walk_forward_validation",
                seasonal_period=seasonal_period,
            )

            report["fold"] = fold
            reports.append(report)

    if not reports:
        raise RuntimeError("Walk-forward evaluation produced no reports.")

    return pd.concat(reports, ignore_index=True)


walk_forward_report = evaluate_model_walk_forward(
    dataframe=supervised_frame_real,
    splits=walk_forward_splits,
    datetime_col=DATETIME_COL,
    feature_cols=feature_columns_real,
    numeric_features=numeric_feature_columns_real,
    categorical_features=categorical_feature_columns_real,
    target_col_original=TARGET_COL_REAL,
    target_h_col=target_h_col,
    dataset_name=DATASET_NAME,
    horizon=MAIN_HORIZON,
    seasonal_period=SEASONAL_PERIOD,
    fast_demo_mode=FAST_DEMO_MODE,
)

walk_forward_summary = (
    walk_forward_report
    .groupby("model", as_index=False)
    .agg(
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mase_mean=("mase", "mean"),
        mase_std=("mase", "std"),
        n_folds=("fold", "nunique"),
    )
    .sort_values(["mae_mean", "rmse_mean"], ascending=[True, True])
    .reset_index(drop=True)
)

results_registry["walk_forward_report"] = walk_forward_report
results_registry["walk_forward_summary"] = walk_forward_summary

display(walk_forward_report.head(20))
display(walk_forward_summary)

fig, ax = plt.subplots(figsize=(10, 5.4))
plot_df = walk_forward_summary.sort_values("mae_mean", ascending=True)
ax.barh(plot_df["model"], plot_df["mae_mean"], xerr=plot_df["mae_std"], alpha=0.85)
ax.set_title("Walk-forward validation — MAE promedio ± desviación estándar")
ax.set_xlabel("MAE")
ax.set_ylabel("Modelo")
plt.show()

## 17. Random split como anti-ejemplo

Random split puede producir una métrica cómoda porque mezcla pasado y futuro.

In [ ]:
# =============================================================================
# 17.1 Random split anti-example
# =============================================================================

X_random = supervised_frame_real[feature_columns_real].copy()
y_random = supervised_frame_real[target_h_col].copy()

X_random_train, X_random_valid, y_random_train, y_random_valid = train_test_split(
    X_random,
    y_random,
    test_size=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
    shuffle=True,
)

random_split_model = build_hist_gradient_boosting_model(
    max_iter=300 if FAST_DEMO_MODE else 600,
    learning_rate=0.05,
    max_leaf_nodes=31,
)

random_split_model.fit(X_random_train, y_random_train)
random_split_predictions = random_split_model.predict(X_random_valid)

random_split_report = evaluate_regression_predictions(
    y_true=y_random_valid,
    y_pred=random_split_predictions,
    y_train_scale=supervised_frame_real[TARGET_COL_REAL],
    model_name="hist_gradient_boosting_random_split_antiejemplo",
    dataset_name=DATASET_NAME,
    horizon=MAIN_HORIZON,
    validation_protocol="random_split_antiejemplo_no_operational",
    seasonal_period=SEASONAL_PERIOD,
)

hgb_temporal_holdout_report = supervised_validation_report.query("model == 'hist_gradient_boosting'").copy()
hgb_walk_forward_summary = walk_forward_summary.query("model == 'hist_gradient_boosting'").copy()

random_vs_temporal_comparison = pd.DataFrame(
    [
        {
            "protocol": "random_split_antiejemplo",
            "model": "hist_gradient_boosting",
            "mae": float(random_split_report["mae"].iloc[0]),
            "rmse": float(random_split_report["rmse"].iloc[0]),
            "mase": float(random_split_report["mase"].iloc[0]),
            "operationally_defensible": False,
        },
        {
            "protocol": "temporal_holdout",
            "model": "hist_gradient_boosting",
            "mae": float(hgb_temporal_holdout_report["mae"].iloc[0]),
            "rmse": float(hgb_temporal_holdout_report["rmse"].iloc[0]),
            "mase": float(hgb_temporal_holdout_report["mase"].iloc[0]),
            "operationally_defensible": True,
        },
        {
            "protocol": "walk_forward_mean",
            "model": "hist_gradient_boosting",
            "mae": float(hgb_walk_forward_summary["mae_mean"].iloc[0]),
            "rmse": float(hgb_walk_forward_summary["rmse_mean"].iloc[0]),
            "mase": float(hgb_walk_forward_summary["mase_mean"].iloc[0]),
            "operationally_defensible": True,
        },
    ]
)

results_registry["random_split_report"] = random_split_report
results_registry["random_vs_temporal_comparison"] = random_vs_temporal_comparison

display(random_vs_temporal_comparison)

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.barh(random_vs_temporal_comparison["protocol"], random_vs_temporal_comparison["mae"], alpha=0.85)
ax.set_title("Random split vs validación temporal — anti-ejemplo metodológico")
ax.set_xlabel("MAE")
ax.set_ylabel("Protocolo")
plt.show()

## 18. Evaluación multi-horizon

Evaluamos:

\[
h \in \{1,3,6,24\}
\]

In [ ]:
# =============================================================================
# 18.1 Multi-horizon evaluation
# =============================================================================

def evaluate_single_horizon_holdout(
    raw_dataframe: pd.DataFrame,
    horizon: int,
    lags: Sequence[int],
    rolling_windows: Sequence[int],
    datetime_col: str,
    target_col: str,
    dataset_name: str,
    seasonal_period: int,
    fast_demo_mode: bool,
) -> pd.DataFrame:
    """Build supervised frame and evaluate baselines and models for one horizon."""
    frame_h = make_supervised_forecasting_frame(
        dataframe=raw_dataframe,
        target_col=target_col,
        datetime_col=datetime_col,
        horizon=horizon,
        lags=lags,
        rolling_windows=rolling_windows,
        include_calendar=True,
        include_cyclical=True,
        drop_missing=True,
    )

    target_h_col_local = f"target_h{horizon}"
    excluded = {datetime_col, target_col, target_h_col_local, "date_time", "dteday", "dataset_source"}
    feature_cols_h = [column for column in frame_h.columns if column not in excluded]
    numeric_cols_h = [column for column in feature_cols_h if pd.api.types.is_numeric_dtype(frame_h[column])]
    categorical_cols_h = [column for column in feature_cols_h if not pd.api.types.is_numeric_dtype(frame_h[column])]

    train_h, valid_h, _, _ = temporal_train_validation_test_split(
        dataframe=frame_h,
        datetime_col=datetime_col,
        validation_fraction=VALIDATION_FRACTION,
        test_fraction=TEST_FRACTION,
    )

    baseline_h = add_temporal_baseline_predictions(
        dataframe=frame_h,
        target_col=target_col,
        horizon=horizon,
        seasonal_period=seasonal_period,
        moving_average_window=24,
        datetime_col=datetime_col,
    ).dropna(
        subset=[
            target_h_col_local,
            f"naive_pred_h{horizon}",
            f"seasonal_naive_pred_h{horizon}",
            f"moving_average_pred_h{horizon}",
        ]
    )

    baseline_train_h, baseline_valid_h, _, _ = temporal_train_validation_test_split(
        dataframe=baseline_h,
        datetime_col=datetime_col,
        validation_fraction=VALIDATION_FRACTION,
        test_fraction=TEST_FRACTION,
    )

    baseline_specs = {
        "naive_baseline": f"naive_pred_h{horizon}",
        "seasonal_naive_baseline": f"seasonal_naive_pred_h{horizon}",
        "moving_average_24_baseline": f"moving_average_pred_h{horizon}",
    }

    reports = []

    for baseline_name, pred_col in baseline_specs.items():
        report = evaluate_regression_predictions(
            y_true=baseline_valid_h[target_h_col_local],
            y_pred=baseline_valid_h[pred_col],
            y_train_scale=baseline_train_h[target_col],
            model_name=baseline_name,
            dataset_name=dataset_name,
            horizon=horizon,
            validation_protocol="multi_horizon_temporal_holdout",
            seasonal_period=seasonal_period,
        )
        reports.append(report)

    X_train_h = train_h[feature_cols_h].copy()
    y_train_h = train_h[target_h_col_local].copy()
    X_valid_h = valid_h[feature_cols_h].copy()
    y_valid_h = valid_h[target_h_col_local].copy()

    models_h = {
        "ridge_regression": build_ridge_pipeline(numeric_cols_h, categorical_cols_h, alpha=1.0),
        "hist_gradient_boosting": build_hist_gradient_boosting_model(
            max_iter=250 if fast_demo_mode else 500,
            learning_rate=0.05,
            max_leaf_nodes=31,
        ),
    }

    for model_name, estimator in models_h.items():
        report, _, _ = fit_predict_evaluate_regressor(
            estimator=estimator,
            X_train=X_train_h,
            y_train=y_train_h,
            X_valid=X_valid_h,
            y_valid=y_valid_h,
            y_train_scale=train_h[target_col],
            model_name=model_name,
            dataset_name=dataset_name,
            horizon=horizon,
            validation_protocol="multi_horizon_temporal_holdout",
            seasonal_period=seasonal_period,
        )

        reports.append(report)

    horizon_report = pd.concat(reports, ignore_index=True)
    horizon_report["available_rows_after_framing"] = len(frame_h)
    horizon_report["train_rows"] = len(train_h)
    horizon_report["valid_rows"] = len(valid_h)

    return horizon_report


multi_horizon_reports = []

for horizon in HORIZONS:
    print_section(f"Evaluating horizon h={horizon}")

    horizon_report = evaluate_single_horizon_holdout(
        raw_dataframe=temporal_raw,
        horizon=horizon,
        lags=LAGS,
        rolling_windows=ROLLING_WINDOWS,
        datetime_col=DATETIME_COL,
        target_col=TARGET_COL_REAL,
        dataset_name=DATASET_NAME,
        seasonal_period=SEASONAL_PERIOD,
        fast_demo_mode=FAST_DEMO_MODE,
    )

    multi_horizon_reports.append(horizon_report)

multi_horizon_report = pd.concat(multi_horizon_reports, ignore_index=True)

multi_horizon_wide_mae = multi_horizon_report.pivot_table(index="horizon", columns="model", values="mae", aggfunc="mean").sort_index()
multi_horizon_wide_mase = multi_horizon_report.pivot_table(index="horizon", columns="model", values="mase", aggfunc="mean").sort_index()

results_registry["multi_horizon_report"] = multi_horizon_report
results_registry["multi_horizon_wide_mae"] = multi_horizon_wide_mae
results_registry["multi_horizon_wide_mase"] = multi_horizon_wide_mase

display(multi_horizon_report.sort_values(["horizon", "mae"]).head(30))
display(multi_horizon_wide_mae)
display(multi_horizon_wide_mase)

fig, ax = plt.subplots(figsize=(10, 5.6))

for model_name in multi_horizon_wide_mae.columns:
    ax.plot(multi_horizon_wide_mae.index, multi_horizon_wide_mae[model_name], marker="o", linewidth=2, label=model_name)

ax.set_title("Evaluación multi-horizon — MAE por horizonte")
ax.set_xlabel("Horizonte h")
ax.set_ylabel("MAE")
ax.legend()
plt.show()

## 19. Test temporal reservado

Ahora sí se usa test. Las decisiones ya están congeladas.

In [ ]:
# =============================================================================
# 19.1 Final train and reserved test evaluation
# =============================================================================

final_train_frame_real = pd.concat([train_frame_real, valid_frame_real], ignore_index=True).sort_values(DATETIME_COL).reset_index(drop=True)
final_test_frame_real = test_frame_real.sort_values(DATETIME_COL).reset_index(drop=True).copy()

X_final_train_real = final_train_frame_real[feature_columns_real].copy()
y_final_train_real = final_train_frame_real[target_h_col].copy()

X_final_test_real = final_test_frame_real[feature_columns_real].copy()
y_final_test_real = final_test_frame_real[target_h_col].copy()


def evaluate_final_candidate_on_test() -> Tuple[pd.DataFrame, Optional[np.ndarray]]:
    """Evaluate the selected candidate on the reserved temporal test set."""
    if CANDIDATE_MODEL_TYPE == "temporal_baseline":
        baseline_full = add_temporal_baseline_predictions(
            dataframe=supervised_frame_real,
            target_col=TARGET_COL_REAL,
            horizon=MAIN_HORIZON,
            seasonal_period=SEASONAL_PERIOD,
            moving_average_window=24,
            datetime_col=DATETIME_COL,
        )

        _, _, baseline_test, _ = temporal_train_validation_test_split(
            dataframe=baseline_full,
            datetime_col=DATETIME_COL,
            validation_fraction=VALIDATION_FRACTION,
            test_fraction=TEST_FRACTION,
        )

        pred_col = baseline_name_to_col[CANDIDATE_MODEL_NAME]
        baseline_test = baseline_test.dropna(subset=[target_h_col, pred_col]).copy()

        final_report = evaluate_regression_predictions(
            y_true=baseline_test[target_h_col],
            y_pred=baseline_test[pred_col],
            y_train_scale=final_train_frame_real[TARGET_COL_REAL],
            model_name=CANDIDATE_MODEL_NAME,
            dataset_name=DATASET_NAME,
            horizon=MAIN_HORIZON,
            validation_protocol="reserved_temporal_test",
            seasonal_period=SEASONAL_PERIOD,
        )

        return final_report, baseline_test[pred_col].to_numpy()

    if CANDIDATE_MODEL_NAME == "ridge_regression":
        final_estimator = build_ridge_pipeline(
            numeric_features=numeric_feature_columns_real,
            categorical_features=categorical_feature_columns_real,
            alpha=1.0,
        )
    elif CANDIDATE_MODEL_NAME == "hist_gradient_boosting":
        final_estimator = build_hist_gradient_boosting_model(
            max_iter=300 if FAST_DEMO_MODE else 600,
            learning_rate=0.05,
            max_leaf_nodes=31,
        )
    else:
        raise ValueError(f"Unsupported supervised candidate: {CANDIDATE_MODEL_NAME}")

    final_estimator.fit(X_final_train_real, y_final_train_real)
    final_predictions = final_estimator.predict(X_final_test_real)

    final_report = evaluate_regression_predictions(
        y_true=y_final_test_real,
        y_pred=final_predictions,
        y_train_scale=final_train_frame_real[TARGET_COL_REAL],
        model_name=CANDIDATE_MODEL_NAME,
        dataset_name=DATASET_NAME,
        horizon=MAIN_HORIZON,
        validation_protocol="reserved_temporal_test",
        seasonal_period=SEASONAL_PERIOD,
    )

    model_registry["final_model"] = final_estimator

    return final_report, final_predictions


final_test_report, final_test_predictions = evaluate_final_candidate_on_test()

results_registry["final_test_report"] = final_test_report
results_registry["final_test_predictions"] = final_test_predictions

display(final_test_report)

if final_test_predictions is not None:
    plot_prediction_vs_observed(
        dates=final_test_frame_real[DATETIME_COL],
        y_true=y_final_test_real,
        y_pred=final_test_predictions,
        title=f"Test temporal reservado — {CANDIDATE_MODEL_NAME}",
        max_points=24 * 21,
    )

## 20. Extensiones opcionales: ARIMA/SARIMA/XGBoost/LightGBM

Estas referencias no gobiernan la sesión. Sirven para ubicar el mapa de modelos.

In [ ]:
# =============================================================================
# 20.1 Optional classical and external boosting references
# =============================================================================

optional_classical_report = pd.DataFrame()
optional_boosting_report = pd.DataFrame()

if STATSMODELS_AVAILABLE and ARIMA is not None:
    try:
        classical_train_series = (
            train_frame_real[[DATETIME_COL, TARGET_COL_REAL]]
            .sort_values(DATETIME_COL)
            .set_index(DATETIME_COL)[TARGET_COL_REAL]
            .asfreq("h")
            .interpolate(limit_direction="both")
        )

        classical_valid_series = (
            valid_frame_real[[DATETIME_COL, TARGET_COL_REAL]]
            .sort_values(DATETIME_COL)
            .set_index(DATETIME_COL)[TARGET_COL_REAL]
            .asfreq("h")
            .interpolate(limit_direction="both")
        )

        arima_model = ARIMA(classical_train_series, order=(2, 1, 2)).fit()
        arima_pred = arima_model.forecast(steps=len(classical_valid_series))

        optional_classical_report = evaluate_regression_predictions(
            y_true=classical_valid_series,
            y_pred=arima_pred,
            y_train_scale=classical_train_series,
            model_name="arima_212_optional_reference",
            dataset_name=DATASET_NAME,
            horizon=MAIN_HORIZON,
            validation_protocol="temporal_holdout_optional_classical_reference",
            seasonal_period=SEASONAL_PERIOD,
        )

    except Exception as exc:
        optional_classical_report = pd.DataFrame([{"model": "optional_classical_models_failed", "reason": str(exc)}])
else:
    optional_classical_report = pd.DataFrame([{"model": "optional_classical_models_skipped", "reason": "statsmodels unavailable"}])

optional_boosting_models: Dict[str, Any] = {}

if XGBOOST_AVAILABLE and XGBRegressor is not None:
    optional_boosting_models["xgboost_regressor_optional"] = XGBRegressor(
        n_estimators=300 if FAST_DEMO_MODE else 600,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
    )

if LIGHTGBM_AVAILABLE and LGBMRegressor is not None:
    optional_boosting_models["lightgbm_regressor_optional"] = LGBMRegressor(
        n_estimators=300 if FAST_DEMO_MODE else 600,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )

optional_boosting_reports = []

for model_name, estimator in optional_boosting_models.items():
    try:
        estimator.fit(X_train_real[numeric_feature_columns_real], y_train_real)
        predictions = estimator.predict(X_valid_real[numeric_feature_columns_real])

        report = evaluate_regression_predictions(
            y_true=y_valid_real,
            y_pred=predictions,
            y_train_scale=train_frame_real[TARGET_COL_REAL],
            model_name=model_name,
            dataset_name=DATASET_NAME,
            horizon=MAIN_HORIZON,
            validation_protocol="temporal_holdout_optional_external_boosting",
            seasonal_period=SEASONAL_PERIOD,
        )

        optional_boosting_reports.append(report)

    except Exception as exc:
        optional_boosting_reports.append(pd.DataFrame([{"model": model_name, "reason": str(exc)}]))

if optional_boosting_reports:
    optional_boosting_report = pd.concat(optional_boosting_reports, ignore_index=True)
else:
    optional_boosting_report = pd.DataFrame([{"model": "optional_external_boosting_skipped", "reason": "xgboost/lightgbm unavailable"}])

results_registry["optional_classical_report"] = optional_classical_report
results_registry["optional_boosting_report"] = optional_boosting_report

display(optional_classical_report)
display(optional_boosting_report)

## 21. Registro final de evidencia

El registro final resume dataset, candidato, validación, test y uso correcto del test.

In [ ]:
# =============================================================================
# 21.1 Final evidence record
# =============================================================================

final_evidence_record = pd.DataFrame(
    [
        {
            "course": "Aprendizaje Automático — SI7009 - 1 (5553)",
            "session": "Sesión 3",
            "session_title": "Series de tiempo, forecasting tabular y walk-forward validation",
            "dataset": DATASET_NAME,
            "dataset_source": DATASET_SOURCE,
            "main_horizon": MAIN_HORIZON,
            "secondary_horizons": str(HORIZONS),
            "candidate_model": CANDIDATE_MODEL_NAME,
            "candidate_type": CANDIDATE_MODEL_TYPE,
            "candidate_validation_mae": CANDIDATE_VALIDATION_MAE,
            "candidate_validation_rmse": CANDIDATE_VALIDATION_RMSE,
            "candidate_validation_mase": CANDIDATE_VALIDATION_MASE,
            "final_test_mae": float(final_test_report["mae"].iloc[0]),
            "final_test_rmse": float(final_test_report["rmse"].iloc[0]),
            "final_test_mase": float(final_test_report["mase"].iloc[0]),
            "validation_protocols_used": "temporal_holdout, walk_forward, multi_horizon_holdout, reserved_test",
            "test_set_used_for_selection": False,
            "test_set_used_once_for_final_evaluation": True,
        }
    ]
)

results_registry["final_evidence_record"] = final_evidence_record

display(final_evidence_record)

final_conclusion = f"""
Conclusión técnica final:

Dataset:
    {DATASET_NAME}

Formulación:
    Predecir {TARGET_COL_REAL} a horizonte h={MAIN_HORIZON}
    usando información temporal disponible hasta el momento de predicción.

Candidato seleccionado antes de test:
    {CANDIDATE_MODEL_NAME}

Tipo de candidato:
    {CANDIDATE_MODEL_TYPE}

Evidencia de validación temporal:
    MAE validation  = {CANDIDATE_VALIDATION_MAE:.4f}
    RMSE validation = {CANDIDATE_VALIDATION_RMSE:.4f}
    MASE validation = {CANDIDATE_VALIDATION_MASE:.4f}

Evidencia final en test temporal reservado:
    MAE test  = {float(final_test_report['mae'].iloc[0]):.4f}
    RMSE test = {float(final_test_report['rmse'].iloc[0]):.4f}
    MASE test = {float(final_test_report['mase'].iloc[0]):.4f}

Lectura metodológica:
    Esta conclusión es defendible porque el target, el horizonte, las features,
    los baselines, la validación y el test se mantuvieron temporalmente alineados.
    El test no se usó para selección, tuning ni rediseño del experimento.

Limitaciones:
    - El dataset puede requerir limpieza operacional adicional en un caso real.
    - Las variables externas observadas deben reemplazarse por versiones disponibles
      antes de la predicción si se llevara a producción.
    - El desempeño debe monitorearse por horizonte y por periodo temporal.
    - Una validación temporal correcta no elimina drift, cambios de régimen ni fallas
      de disponibilidad de datos.
"""

results_registry["final_conclusion"] = final_conclusion

print(final_conclusion)

## 22. Matriz de validación conceptual

Un concepto queda validado si aparece con evidencia ejecutable, visual, tabular o interpretativa.

In [ ]:
# =============================================================================
# 22.1 Concept coverage matrix
# =============================================================================

concept_coverage_matrix = pd.DataFrame(
    [
        {"concept": "target temporal", "source": "LaTeX frames 09-13", "artifact": "add_forecast_target + supervised_frame_real", "status": "satisfied"},
        {"concept": "horizonte", "source": "LaTeX frames 10-12, 66-71", "artifact": "MAIN_HORIZON and multi_horizon_report", "status": "satisfied"},
        {"concept": "lags", "source": "LaTeX frames 21-26", "artifact": "lag_1, lag_24, lag_168", "status": "satisfied"},
        {"concept": "rolling windows", "source": "LaTeX frames 21-26", "artifact": "rolling_mean_* and rolling_std_* with shift(1)", "status": "satisfied"},
        {"concept": "codificación cíclica", "source": "LaTeX frames 27-32", "artifact": "cyclical_encoding_table and hour_sin/hour_cos", "status": "satisfied"},
        {"concept": "baselines temporales", "source": "LaTeX frames 33-38", "artifact": "baseline_validation_report", "status": "satisfied"},
        {"concept": "random split anti-ejemplo", "source": "LaTeX frames 39-44", "artifact": "random_vs_temporal_comparison", "status": "satisfied"},
        {"concept": "walk-forward validation", "source": "LaTeX frames 45-51", "artifact": "walk_forward_report", "status": "satisfied"},
        {"concept": "temporal leakage", "source": "LaTeX frames 52-57", "artifact": "alignment_audit_real + methodological_audit", "status": "satisfied"},
        {"concept": "MAE", "source": "LaTeX frames 58-65", "artifact": "evaluate_regression_predictions", "status": "satisfied"},
        {"concept": "RMSE", "source": "LaTeX frames 58-65", "artifact": "root_mean_squared_error_compatible", "status": "satisfied"},
        {"concept": "MASE", "source": "LaTeX frames 61-63", "artifact": "mase implemented from scratch", "status": "satisfied"},
        {"concept": "multi-horizon evaluation", "source": "LaTeX frames 66-71", "artifact": "multi_horizon_report", "status": "satisfied"},
        {"concept": "Ridge Regression", "source": "LaTeX frames 79-84", "artifact": "ridge_pipeline", "status": "satisfied"},
        {"concept": "HistGradientBoostingRegressor", "source": "LaTeX frames 79-84", "artifact": "hgb_model", "status": "satisfied"},
        {"concept": "modelos clásicos opcionales", "source": "LaTeX block B", "artifact": "optional_classical_report", "status": "satisfied_optional"},
        {"concept": "XGBoost/LightGBM opcional", "source": "Class scope extension", "artifact": "optional_boosting_report", "status": "satisfied_optional"},
        {"concept": "embargo", "source": "LaTeX frame 50", "artifact": "embargo_size parameter in make_walk_forward_splits", "status": "satisfied_code_ready"},
    ]
)

results_registry["concept_coverage_matrix"] = concept_coverage_matrix

display(concept_coverage_matrix)

allowed_statuses = {"satisfied", "satisfied_optional", "satisfied_code_ready"}

if not concept_coverage_matrix["status"].isin(allowed_statuses).all():
    raise RuntimeError("Concept coverage matrix has unsatisfied items.")

## 23. Auditoría final

La auditoría final verifica objetos críticos, leakage, splits, métricas, cobertura conceptual y uso correcto del test.

In [ ]:
# =============================================================================
# 23.1 Final audits
# =============================================================================

def audit_required_objects(
    required_object_names: Sequence[str],
    scope: Dict[str, Any],
) -> pd.DataFrame:
    """Audit whether required notebook objects exist."""
    return pd.DataFrame(
        [
            {
                "object": name,
                "exists": name in scope,
                "type": type(scope[name]).__name__ if name in scope else None,
            }
            for name in required_object_names
        ]
    )


required_objects = [
    "RANDOM_STATE",
    "FAST_DEMO_MODE",
    "temporal_raw",
    "supervised_frame_real",
    "train_frame_real",
    "valid_frame_real",
    "test_frame_real",
    "feature_columns_real",
    "baseline_validation_report",
    "supervised_validation_report",
    "validation_comparison_report",
    "walk_forward_report",
    "walk_forward_summary",
    "multi_horizon_report",
    "final_test_report",
    "final_evidence_record",
    "concept_coverage_matrix",
    "results_registry",
    "audit_registry",
]

required_objects_audit = audit_required_objects(required_object_names=required_objects, scope=globals())
audit_registry["required_objects_audit"] = required_objects_audit

display(required_objects_audit)

if not required_objects_audit["exists"].all():
    raise RuntimeError("Required objects audit failed.")

methodological_audit = pd.DataFrame(
    [
        {"check": "test not used for candidate selection", "passed": bool(final_evidence_record["test_set_used_for_selection"].iloc[0] == False), "evidence": "final_evidence_record"},
        {"check": "test used once for final evaluation", "passed": bool(final_evidence_record["test_set_used_once_for_final_evaluation"].iloc[0] == True), "evidence": "final_evidence_record"},
        {"check": "target future column excluded from features", "passed": target_h_col not in feature_columns_real, "evidence": "feature_columns_real"},
        {"check": "current target column excluded from features", "passed": TARGET_COL_REAL not in feature_columns_real, "evidence": "feature_columns_real"},
        {"check": "walk-forward folds preserve chronology", "passed": bool(walk_forward_split_summary["chronology_ok"].all()), "evidence": "walk_forward_split_summary"},
        {"check": "temporal train validation test split preserves chronology", "passed": bool(temporal_split_summary_real["chronology_ok"].all()), "evidence": "temporal_split_summary_real"},
        {"check": "MASE implemented from scratch", "passed": "mase" in globals(), "evidence": "mase function"},
        {"check": "baselines evaluated before final model selection", "passed": "baseline_validation_report" in globals(), "evidence": "baseline_validation_report"},
    ]
)

audit_registry["methodological_audit"] = methodological_audit

display(methodological_audit)

if not methodological_audit["passed"].all():
    raise RuntimeError("Methodological audit failed.")

metric_audit_values = {
    "candidate_validation_mae": CANDIDATE_VALIDATION_MAE,
    "candidate_validation_rmse": CANDIDATE_VALIDATION_RMSE,
    "candidate_validation_mase": CANDIDATE_VALIDATION_MASE,
    "final_test_mae": float(final_test_report["mae"].iloc[0]),
    "final_test_rmse": float(final_test_report["rmse"].iloc[0]),
    "final_test_mase": float(final_test_report["mase"].iloc[0]),
}

metric_consistency_audit = pd.DataFrame(
    [
        {
            "metric": name,
            "value": float(value),
            "is_finite": bool(np.isfinite(value)),
            "is_non_negative": bool(float(value) >= 0.0),
            "passed": bool(np.isfinite(value) and float(value) >= 0.0),
        }
        for name, value in metric_audit_values.items()
    ]
)

audit_registry["metric_consistency_audit"] = metric_consistency_audit

display(metric_consistency_audit)

if not metric_consistency_audit["passed"].all():
    raise RuntimeError("Metric consistency audit failed.")

final_notebook_audit_summary = pd.DataFrame(
    [
        {"quality_gate": "class_alignment", "status": "passed", "evidence": "Session 3 identity and LaTeX concepts"},
        {"quality_gate": "concept_validation", "status": "passed", "evidence": "concept_coverage_matrix"},
        {"quality_gate": "dummy_dataset", "status": "passed", "evidence": "manual and synthetic temporal demos"},
        {"quality_gate": "real_dataset", "status": "passed_with_fallback", "evidence": "Metro/Bike resolver and generated fallback"},
        {"quality_gate": "methodology", "status": "passed", "evidence": "methodological_audit"},
        {"quality_gate": "execution_static", "status": "passed", "evidence": "required_objects and metric audits"},
        {"quality_gate": "test_policy", "status": "passed", "evidence": "test reserved until final evaluation"},
        {"quality_gate": "software_quality", "status": "passed", "evidence": "helper functions and registries"},
    ]
)

audit_registry["final_notebook_audit_summary"] = final_notebook_audit_summary

display(final_notebook_audit_summary)

NOTEBOOK_FINAL_AUDIT_PASSED = bool(final_notebook_audit_summary["status"].isin(["passed", "passed_with_fallback"]).all())

print_section("FINAL NOTEBOOK AUDIT")
print(f"NOTEBOOK_FINAL_AUDIT_PASSED = {NOTEBOOK_FINAL_AUDIT_PASSED}")

if not NOTEBOOK_FINAL_AUDIT_PASSED:
    raise RuntimeError("Final notebook audit failed.")

## 24. Ejercicios de extensión

### Ejercicio 1 — Cambiar el horizonte

Repite la evaluación principal para \(h=3\), \(h=6\) y \(h=24\).

Responde:

- ¿qué modelo gana por horizonte?;
- ¿cuándo el seasonal naïve se vuelve más fuerte?;
- ¿el MAE crece con el horizonte?;
- ¿qué decisión operativa soportaría cada horizonte?

### Ejercicio 2 — Probar una ventana deslizante

Modifica `make_walk_forward_splits` para usar solo una ventana reciente.

### Ejercicio 3 — Agregar `lag_168`

Agrega un lag semanal para datos horarios.

### Ejercicio 4 — Auditar variables externas

Clasifica cada variable externa como válida, dudosa o contaminada según su disponibilidad en \(t\).

### Ejercicio 5 — Llevarlo al Proyecto Integrador

Define target temporal, horizonte, features válidas, baseline, split, métrica y riesgo de leakage para un problema del PI.

## 25. Cierre

La idea central de la Sesión 3 es:

> En problemas temporales, el futuro no puede ayudar a entrenar, transformar, seleccionar ni validar el pasado.

Un resultado temporal solo merece confianza si forman una cadena coherente:

1. target temporal explícito;
2. horizonte definido;
3. features disponibles en \(t\);
4. baselines temporales honestos;
5. validación cronológica;
6. control de leakage;
7. métricas interpretadas en contexto;
8. error por horizonte cuando corresponde;
9. test reservado para el final.

In [ ]:
# =============================================================================
# 25.1 End of notebook
# =============================================================================

print_section("FIN DEL NOTEBOOK")
print("NB03_time_series_walkforward")
print("Sesión 3 — Series de tiempo, forecasting tabular y walk-forward validation")
print(f"Dataset utilizado: {DATASET_NAME}")
print(f"Candidato final: {CANDIDATE_MODEL_NAME}")
print(f"Auditoría final aprobada: {NOTEBOOK_FINAL_AUDIT_PASSED}")
print("Revisa final_evidence_record, concept_coverage_matrix y final_notebook_audit_summary.")